In [2]:
import pandas as pd

In [7]:
# Export clean file for Tableau

sessions_featured = pd.read_csv('../data_csv/4 sessions_featured.csv')
sessions_modeled = pd.read_csv('../data_csv/5 sessions_modeled_lgb.csv')

tableau_df = pd.concat([
    sessions_featured,
    sessions_modeled[['purchase_prob', 'segment']]
], axis=1)

# Add funnel stage column
def funnel_stage(row):
    if row['num_purchase'] > 0:
        return 'Purchase'
    elif row['num_cart'] > 0:
        return 'Cart'
    else:
        return 'View Only'

tableau_df['funnel_stage'] = tableau_df.apply(funnel_stage, axis=1)

tableau_df.to_csv('tableau_data.csv', index=False)

In [3]:
print(tableau_df.columns.tolist())

['user_session', 'user_id', 'session_start', 'session_end', 'total_events', 'num_views', 'num_cart', 'num_purchase', 'num_remove', 'num_returns', 'unique_products', 'avg_price', 'max_price', 'hour', 'day_of_week', 'session_duration_min', 'converted', 'view_depth', 'single_view', 'hesitation_rate', 'cart_abandoned', 'cart_to_purchase_rate', 'purchase_prob', 'segment', 'funnel_stage']


In [15]:
# pic2 Simpler duration bins
duration = tableau_df[
    tableau_df['session_duration_min'].notna() ].copy()

duration['duration_bin'] = pd.cut(
    duration['session_duration_min'], 
    bins=[0, 1, 5, 15, 30, 60, 120, 1440],
    labels=['0-1 min', '1-5 min', '5-15 min', '15-30 min', '30-60 min', '1-2 hours', 'more than 2 hours'])

duration_summary = duration.groupby('duration_bin', observed=True).agg(
    sessions=('user_session', 'count')
).reset_index()

duration_summary.to_csv('7.2 tableau_duration.csv', index=False)
print(duration_summary)

        duration_bin  sessions
0            0-1 min    379297
1            1-5 min    457786
2           5-15 min    331402
3          15-30 min    169110
4          30-60 min    113964
5          1-2 hours     70613
6  more than 2 hours     67755


In [20]:
# Exclude returns and negative prices
price_data = tableau_df[
    (tableau_df['avg_price'] > 0) &
    (tableau_df['num_returns'] == 0)
]['avg_price']

price_summary = pd.DataFrame({
    'Metric': ['P10', 'P25', 'Median', 'Average', 'P75', 'P90', 'Max'],
    'Value': [
        price_data.quantile(0.10).round(2),
        price_data.quantile(0.25).round(2),
        price_data.quantile(0.50).round(2),
        price_data.mean().round(2),
        price_data.quantile(0.75).round(2),
        price_data.quantile(0.90).round(2),
        price_data.max().round(2)
    ]
})
print(price_summary)
price_summary.to_csv('7.4 tableau_price.csv', index=False)

    Metric   Value
0      P10    1.73
1      P25    3.14
2   Median    5.38
3  Average   13.79
4      P75   11.08
5      P90   30.14
6      Max  327.78


In [ ]:
#图6 customer segment
segment_summary = tableau_df.groupby('segment').agg(
    sessions=('user_session', 'count'),
    conversion_rate=('converted', 'mean')
).reset_index()

# 不乘100，保留原始比例，Tableau里再格式化
segment_summary.to_csv('7.6 tableau_segment.csv', index=False)
print(segment_summary)

         segment  sessions  conversion_rate
0    High Intent    453182         0.311182
1     Low Intent   3400295         0.000017
2  Medium Intent    682463         0.021301


In [ ]:

# 图1: Overview
overview = pd.DataFrame({
    'Metric': ['Unique Users', 'Unique Sessions'],
    'Value': [
        tableau_df['user_id'].nunique(),
        tableau_df['user_session'].nunique()
    ]
})
overview.to_csv('7.1 tableau_overview.csv', index=False)


# 图3: Event type distribution (替代category)
event_summary = pd.DataFrame({
    'Event_Type': ['View Only', 'Cart', 'Purchase'],
    'Sessions': [
        (tableau_df['funnel_stage'] == 'View Only').sum(),
        (tableau_df['funnel_stage'] == 'Cart').sum(),
        (tableau_df['funnel_stage'] == 'Purchase').sum()
    ]
})
event_summary.to_csv('7.3 tableau_events.csv', index=False)



# 图5: Funnel
funnel = pd.DataFrame({
    'Stage': ['View', 'Cart', 'Purchase'],
    'Sessions': [4280699, 788430, 106526],
    'Conversion_Rate': [100.0, 18.42, 2.49]
})
funnel.to_csv('7.5 tableau_funnel.csv', index=False)


# 图7: Hourly heatmap
hour_day = tableau_df.groupby(['hour', 'day_of_week']).agg(
    conversion_rate=('converted', 'mean')
).reset_index()
hour_day['conversion_rate'] = hour_day['conversion_rate'] * 100
hour_day.to_csv('7.7 tableau_hourday.csv', index=False)



In [ ]:
# 图8: Behavioral diagnosis
diagnosis = pd.DataFrame({
    'Feature': ['view_depth', 'total_events', 'unique_products', 'single_view',
                'session_duration_min', 'num_cart', 'num_views', 'hesitation_rate', 'avg_price'],
    'Converted': [4.10, 27.59, 14.75, 0.167, 35.36, 7.76, 6.61, 0.447, 7.41],
    'Not_Converted': [3.02, 20.59, 13.17, 0.144, 42.22, 8.49, 7.29, 0.426, 7.12],
    'Difference_Pct': [35.9, 34.0, 12.1, 15.9, -16.2, -8.6, -9.3, 5.0, 4.1]
})
diagnosis.to_csv('7.8 tableau_diagnosis.csv', index=False)

In [11]:
# Load events data
events = pd.read_csv('../data_csv/2 events_cleaned.csv', usecols=['category_id', 'brand', 'event_type'])

# How many unique categories
print(f"Unique categories: {events['category_id'].nunique():,}")

# Brand analysis
print(f"\nTotal unknown brand: {(events['brand'] == 'unknown').sum():,}")
print(f"Unknown brand %: {(events['brand'] == 'unknown').mean():.1%}")

# Top 5 known brands by events
top_brands = events[events['brand'] != 'unknown']['brand'].value_counts().head(5)
print(f"\nTop 5 known brands:")
print(top_brands)

# Top 5 categories by events
top_cats = events['category_id'].value_counts().head(5)
print(f"\nTop 5 categories:")
print(top_cats)

Unique categories: 525

Total unknown brand: 8,263,663
Unknown brand %: 42.2%

Top 5 known brands:
brand
runail       1433792
irisk         970331
grattol       811021
masura        801532
bpw.style     409838
Name: count, dtype: int64

Top 5 categories:
category_id
1487580007675986893    996973
1487580005092295511    734233
1487580005595612013    731909
1487580005671109489    627179
1602943681873052386    610700
Name: count, dtype: int64


In [14]:
# Brand pie chart: top 5 known + unknown + others
brand_counts = events['brand'].value_counts()

top5_known = brand_counts[brand_counts.index != 'unknown'].head(5)
unknown = brand_counts['unknown']
others = brand_counts[~brand_counts.index.isin(top5_known.index) & 
                      (brand_counts.index != 'unknown')].sum()

brand_pie = pd.DataFrame({
    'brand': list(top5_known.index) + ['unknown', 'others'],
    'events': list(top5_known.values) + [unknown, others]
})

print(brand_pie)
brand_pie.to_csv('7.9 tableau_brands_pie.csv', index=False)

       brand   events
0     runail  1433792
1      irisk   970331
2    grattol   811021
3     masura   801532
4  bpw.style   409838
5    unknown  8263663
6     others  6889474
